# Item-mismatched control: logit decomposition

**Purpose.** Run the 50-perturbation item-mismatched (random-source) control at each model's triplet-selected hook+layer, but log per-perturbation per-item `target_logit_delta` and `distractor_logit_delta`. Existing `mechanism_random_controls.csv` only stores a scalar recovery R, which cannot be decomposed into the two logit channels.

**Question this answers:** Under fully item-mismatched source activations (no identity match, no local value), is the distractor logit still suppressed?
- If YES → distractor suppression is a generic patching side-effect; the matched triplet's unique action is **target-logit protection**.
- If NO → distractor suppression correlates with item-identity match, supporting the current §5.3 framing.

**Self-contained.** This notebook installs its own deps, defines all helpers inline, and reads `mechanism_summary.csv` (from the existing repro run) to look up each model's triplet-selected hook+layer (skipping the expensive layer rescan).

**Prerequisites.**
1. GPU runtime (single A100 recommended). CPU works but slow.
2. Hugging Face token (for gated Gemma models). Set `HF_TOKEN` below or via env / Colab secrets.
3. The existing `mechanism_summary.csv` from the main reproduction run. Set `MECHANISM_SUMMARY_PATH` below. If you don't have one, this notebook can run the layer scan internally too (set `MECHANISM_SUMMARY_PATH = None`).

**Output.** `mechanism_random_controls_logit_decomp.csv` — schema mirrors `mechanism_definition_target_swap_controls.csv`:
```
model_key, model, item_id, word, target, distractor, neutral_word,
hook_name, triplet_selected_layer, control_type='item_mismatched_clean',
perm, donor_perm_index, donor_item_id, donor_word, donor_target,
m_clean, m_corrupt, m_matched, m_patched,
target_logit_delta, distractor_logit_delta
```

**Cost.** ~30 items × (1 matched + 50 random) × 5 models ≈ 250 forward passes per model on top of model loading + clean/corrupt caching. ~10–25 min per model, ~1.5 h total on A100.

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
import sys, subprocess, pkgutil

def ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name.replace('-', '_')
    if pkgutil.find_loader(import_name) is None:
        print(f'Installing {pip_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

for pkg, imp in [
    ('transformers>=4.45.0', 'transformers'),
    ('accelerate', 'accelerate'),
    ('sentencepiece', 'sentencepiece'),
    ('protobuf', 'google.protobuf'),
    ('transformer-lens>=3.0.0', 'transformer_lens'),
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('tqdm', 'tqdm'),
]:
    try:
        ensure_package(pkg, imp)
    except Exception as e:
        print('Install failed/skipped:', pkg, repr(e))
print('Dependencies ready.')

In [ ]:
# ============================================================
# 2. Imports + config
# ============================================================
import os, gc, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from transformer_lens import HookedTransformer

warnings.filterwarnings('ignore')
torch.set_grad_enabled(False)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MECHANISM_DTYPE = torch.float32

# Hyperparameters (match the main reproduction run).
PATCH_N_ITEMS = 32
NEUTRALS_PER_ITEM_MECHANISM = 8
MECHANISM_BATCH_SIZE = 64
N_RANDOM_PERMUTATIONS = 50
MIN_MECHANISM_LAYER = 2
USE_CHAT_TEMPLATE_FOR_INSTRUCT = True

# --- Hugging Face authentication ---
HF_TOKEN = ''  # paste your HF token here, or set env/Colab secret
try:
    from google.colab import userdata
    if not HF_TOKEN:
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print('HF login OK.')
    except Exception as e:
        print('HF login failed:', repr(e))
else:
    print('No HF token; gated Gemma models will fail. Set HF_TOKEN above.')

# --- Paths ---
# Where to save the new CSV. Default: a sibling dir named random_logit_decomp_<timestamp>.
from datetime import datetime
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = Path(f'/content/random_logit_decomp_{RUN_ID}') if Path('/content').exists() else Path(f'./random_logit_decomp_{RUN_ID}')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Path to existing mechanism_summary.csv from the main reproduction run. If None,
# this notebook will run the layer scan internally per model (much slower).
MECHANISM_SUMMARY_PATH = None  # e.g. Path('/content/semantic_stroop_final_reproduction_YYYYMMDD_HHMMSS/csv/mechanism_summary.csv')

print('Device:', DEVICE)
print('Output dir:', OUT_DIR)

In [ ]:
# ============================================================
# 3. Mechanism model specs (5 aligned tractable models)
# ============================================================
MECHANISM_MODEL_SPECS = [
    {'model_key': 'qwen25_15b_base',     'hf_id': 'Qwen/Qwen2.5-1.5B',          'tl_name': 'Qwen/Qwen2.5-1.5B',          'family': 'Qwen',  'params_b': 1.5, 'tuning': 'base',     'enabled': True},
    {'model_key': 'qwen25_15b_instruct', 'hf_id': 'Qwen/Qwen2.5-1.5B-Instruct', 'tl_name': 'Qwen/Qwen2.5-1.5B-Instruct', 'family': 'Qwen',  'params_b': 1.5, 'tuning': 'instruct', 'enabled': True},
    {'model_key': 'gemma2_2b_base',      'hf_id': 'google/gemma-2-2b',          'tl_name': 'google/gemma-2-2b',          'family': 'Gemma', 'params_b': 2.0, 'tuning': 'base',     'enabled': True},
    {'model_key': 'gemma2_2b_instruct',  'hf_id': 'google/gemma-2-2b-it',       'tl_name': 'google/gemma-2-2b-it',       'family': 'Gemma', 'params_b': 2.0, 'tuning': 'instruct', 'enabled': True},
    {'model_key': 'olmo_1b_base',        'hf_id': 'allenai/OLMo-1B-hf',         'tl_name': 'allenai/OLMo-1B-hf',         'family': 'OLMo',  'params_b': 1.0, 'tuning': 'base',     'enabled': True},
]

In [ ]:
# ============================================================
# 4. Stimuli and helpers (inlined from main reproduction notebook)
# ============================================================
MECHANISM_ITEMS = [
    ('small','tiny','big'),('large','big','small'),('hot','warm','cold'),('cold','chilly','hot'),
    ('happy','glad','sad'),('sad','unhappy','happy'),('fast','quick','slow'),('slow','sluggish','fast'),
    ('bright','shiny','dark'),('dark','dim','bright'),('hard','tough','soft'),('soft','gentle','hard'),
    ('strong','powerful','weak'),('weak','frail','strong'),('old','aged','new'),('new','fresh','old'),
    ('clean','neat','dirty'),('dirty','filthy','clean'),('rich','wealthy','poor'),('poor','needy','rich'),
    ('safe','secure','dangerous'),('dangerous','risky','safe'),('easy','simple','hard'),('difficult','hard','easy'),
    ('loud','noisy','quiet'),('quiet','silent','loud'),('clear','plain','confusing'),('confusing','unclear','clear'),
    ('empty','vacant','full'),('full','complete','empty'),('heavy','weighty','light'),('light','bright','dark'),
    ('narrow','thin','wide'),('wide','broad','narrow'),('deep','profound','shallow'),('shallow','superficial','deep'),
    ('near','close','far'),('far','distant','near'),('young','youthful','old'),('rough','coarse','smooth'),
    ('smooth','sleek','rough'),('wet','damp','dry'),('open','unlocked','closed'),('closed','shut','open'),
    ('kind','nice','cruel'),('cruel','mean','kind'),('brave','bold','cowardly'),('calm','peaceful','anxious'),
    ('anxious','nervous','calm'),('honest','truthful','dishonest'),('lazy','idle','active'),('active','busy','lazy'),
    ('complex','complicated','simple'),('cheap','inexpensive','expensive'),('expensive','costly','cheap'),
    ('false','fake','true'),('thick','dense','thin'),('thin','slim','thick'),('sharp','pointed','dull'),
    ('dull','blunt','sharp'),('ugly','hideous','beautiful'),('smart','clever','stupid'),('stupid','dumb','smart'),
    ('bad','poor','good'),('wrong','incorrect','right'),
]
mechanism_stimuli = pd.DataFrame(MECHANISM_ITEMS, columns=['word','distractor','target'])
mechanism_stimuli['item_id'] = np.arange(len(mechanism_stimuli))

NEUTRAL_WORDS = [
    'thing','object','item','label','symbol','marker','term','name','token','entity','unit','case',
    'entry','sample','sign','tag','code','note','point','slot','type','class','record','field',
    'node','file','value','key','piece','part','form','line','phrase','signal','index','number',
    'letter','word','text','rule','box','shape','mark','group','set','map','link','path','step',
    'place','zone','area','page','cell','row','column','frame','scene','view','image',
]

MECHANISM_TEMPLATE = 'Complete the answer with one word only.\nIn this game, {subject} means {meaning}.\nQuestion: In this game, a synonym of {query} is\nAnswer:'

def mechanism_prompt(subject, meaning, query):
    return MECHANISM_TEMPLATE.format(subject=subject, meaning=meaning, query=query)

def choose_mechanism_neutrals(item_idx, item):
    blocked = {item['word'], item['distractor'], item['target']}
    candidates = [w for w in NEUTRAL_WORDS if w not in blocked]
    offset = (item_idx * NEUTRALS_PER_ITEM_MECHANISM) % len(candidates)
    out = []
    for j in range(len(candidates)):
        c = candidates[(offset + j) % len(candidates)]
        if c not in out:
            out.append(c)
        if len(out) >= NEUTRALS_PER_ITEM_MECHANISM:
            break
    return out

def find_subsequence(seq, sub):
    return [i for i in range(len(seq) - len(sub) + 1) if list(seq[i:i+len(sub)]) == list(sub)]

def expand_positions(starts, length):
    out = []
    for s in starts:
        out.extend(range(s, s + length))
    return sorted(set(out))

def recovery(clean, corrupt, patched, idx=None):
    clean = np.asarray(clean, float); corrupt = np.asarray(corrupt, float); patched = np.asarray(patched, float)
    if idx is None:
        idx = np.arange(len(clean))
    idx = np.asarray(idx)
    denom = clean[idx].mean() - corrupt[idx].mean()
    if abs(denom) < 1e-9:
        return float('nan')
    return float((patched[idx].mean() - corrupt[idx].mean()) / denom)

print('Helpers + stimuli ready. mechanism_stimuli shape:', mechanism_stimuli.shape)

In [ ]:
# ============================================================
# 5. Tokenizer + model loading helpers
# ============================================================
def patch_tokenizer_no_bos_if_needed(tokenizer):
    try:
        if getattr(tokenizer, 'bos_token', None) is None and hasattr(tokenizer, 'add_bos_token'):
            tokenizer.add_bos_token = False
    except Exception:
        pass
    try:
        if getattr(tokenizer, 'bos_token_id', None) is None and hasattr(tokenizer, 'add_bos_token'):
            tokenizer.add_bos_token = False
    except Exception:
        pass
    try:
        if getattr(tokenizer, 'pad_token', None) is None:
            tokenizer.pad_token = tokenizer.eos_token
    except Exception:
        pass
    return tokenizer

def load_safe_tokenizer(hf_id):
    attempts = [
        {'trust_remote_code': True, 'use_fast': True, 'add_bos_token': False},
        {'trust_remote_code': True, 'use_fast': False, 'add_bos_token': False},
        {'trust_remote_code': True, 'use_fast': True},
        {'trust_remote_code': True, 'use_fast': False},
    ]
    errors = []
    for kw in attempts:
        try:
            tok = AutoTokenizer.from_pretrained(hf_id, **kw)
            return patch_tokenizer_no_bos_if_needed(tok)
        except TypeError as e:
            errors.append(repr(e))
            kw2 = {k: v for k, v in kw.items() if k != 'add_bos_token'}
            try:
                tok = AutoTokenizer.from_pretrained(hf_id, **kw2)
                return patch_tokenizer_no_bos_if_needed(tok)
            except Exception as e2:
                errors.append(repr(e2))
        except Exception as e:
            errors.append(repr(e))
    raise RuntimeError(f'Failed tokenizer load: {hf_id}: {errors[-3:]}')

def load_hooked_transformer_safe(spec):
    tl_name = spec.get('tl_name', spec['hf_id']); hf_id = spec['hf_id']
    base_kwargs = dict(device=DEVICE, dtype=MECHANISM_DTYPE, default_prepend_bos=False)
    last_error = None
    try:
        return HookedTransformer.from_pretrained(tl_name, tokenizer=load_safe_tokenizer(hf_id), **base_kwargs)
    except Exception as e:
        last_error = e
    try:
        return HookedTransformer.from_pretrained(tl_name, tokenizer_kwargs={'add_bos_token': False, 'trust_remote_code': True}, **base_kwargs)
    except Exception as e:
        last_error = e
    try:
        model = HookedTransformer.from_pretrained(tl_name, **base_kwargs)
        model.tokenizer = patch_tokenizer_no_bos_if_needed(model.tokenizer)
        return model
    except Exception as e:
        last_error = e
    raise last_error

print('Model-loading helpers ready.')

In [ ]:
# ============================================================
# 6. Optional: load existing mechanism_summary.csv to skip layer scan
# ============================================================
summary_df = None
if MECHANISM_SUMMARY_PATH is not None and Path(MECHANISM_SUMMARY_PATH).exists():
    summary_df = pd.read_csv(MECHANISM_SUMMARY_PATH)
    print('Loaded mechanism_summary.csv:')
    print(summary_df[['model_key','best_source_label','best_source_layer','best_source_component','best_source_recovery']].to_string(index=False))
else:
    print('No mechanism_summary.csv found. Will run layer scan internally for each model (slower).')
    print('To skip the scan, set MECHANISM_SUMMARY_PATH above to your main run\\'s mechanism_summary.csv.')

In [ ]:
# ============================================================
# 7. Main worker: build state + run random-controls with logit logging
# ============================================================
RESIDUAL_COMPONENTS = ['hook_resid_pre', 'hook_resid_mid', 'hook_resid_post']

def run_random_logit_decomp_for_model(spec, summary_df=None):
    print('\n' + '=' * 100)
    print('Model:', spec['model_key'], spec.get('tl_name', spec['hf_id']))
    print('=' * 100)

    # --- Load model ---
    model = load_hooked_transformer_safe(spec)
    model.eval()
    model.tokenizer = patch_tokenizer_no_bos_if_needed(model.tokenizer)
    USE_BOS = False
    n_layers = int(model.cfg.n_layers)

    def toks_no_bos(text):
        ids = model.to_tokens(str(text), prepend_bos=False).squeeze(0).tolist()
        if isinstance(ids, int):
            ids = [ids]
        return [int(x) for x in ids]
    def cont_ids(text):
        return toks_no_bos(' ' + str(text).strip())
    def is_single_token(text):
        return len(cont_ids(text)) == 1
    def to_tokens(prompts):
        return model.to_tokens(list(prompts), prepend_bos=USE_BOS).to(DEVICE)
    def maybe_mech_chat_wrap(prompt):
        if USE_CHAT_TEMPLATE_FOR_INSTRUCT and spec.get('tuning') == 'instruct' and hasattr(model.tokenizer, 'apply_chat_template'):
            try:
                return model.tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)
            except Exception:
                return prompt
        return prompt
    def logit_parts(logits_2d, target_ids, distractor_ids):
        logits_2d = logits_2d.float()
        idx = torch.arange(logits_2d.shape[0], device=logits_2d.device)
        t = logits_2d[idx, target_ids]
        d = logits_2d[idx, distractor_ids]
        return (t.detach().cpu().numpy(), d.detach().cpu().numpy(), (t-d).detach().cpu().numpy())

    # --- Build valid single-token items ---
    valid_rows = []
    for _, item in mechanism_stimuli.iterrows():
        if not (is_single_token(item['word']) and is_single_token(item['distractor']) and is_single_token(item['target'])):
            continue
        raw = mechanism_prompt(item['word'], item['target'], item['word'])
        valid_rows.append({
            **item.to_dict(),
            'word_id': cont_ids(item['word'])[0],
            'distractor_id': cont_ids(item['distractor'])[0],
            'target_id': cont_ids(item['target'])[0],
            'conflict_prompt': maybe_mech_chat_wrap(raw),
        })
    valid = pd.DataFrame(valid_rows).reset_index(drop=True)
    if len(valid) < 10:
        raise RuntimeError(f'Too few valid single-token items: {len(valid)}')
    print(f'  valid single-token items: {len(valid)}')

    # --- Conflict + multi-neutral scoring (used to pick representative neutral) ---
    def score_prompts(prompts, tids, dids):
        rows = []
        for st in range(0, len(prompts), MECHANISM_BATCH_SIZE):
            ed = min(st + MECHANISM_BATCH_SIZE, len(prompts))
            toks = to_tokens(prompts[st:ed])
            t = torch.tensor(tids[st:ed], dtype=torch.long, device=DEVICE)
            d = torch.tensor(dids[st:ed], dtype=torch.long, device=DEVICE)
            with torch.inference_mode():
                logits = model(toks)[:, -1, :]
            _, _, diff = logit_parts(logits, t, d)
            rows.extend(list(diff))
            del toks, t, d, logits
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return np.asarray(rows, dtype=float)

    conflict_scores = score_prompts(
        valid['conflict_prompt'].tolist(),
        valid['target_id'].astype(int).tolist(),
        valid['distractor_id'].astype(int).tolist(),
    )
    neutral_rows = []
    neutral_prompts, nt, nd, meta = [], [], [], []
    for i, row in valid.iterrows():
        for nw in choose_mechanism_neutrals(i, row):
            raw = mechanism_prompt(nw, row['target'], nw)
            prompt = maybe_mech_chat_wrap(raw)
            neutral_prompts.append(prompt); nt.append(int(row['target_id'])); nd.append(int(row['distractor_id']))
            meta.append((i, nw, prompt))
    neutral_scores = score_prompts(neutral_prompts, nt, nd)
    for score, (i, nw, prompt) in zip(neutral_scores, meta):
        neutral_rows.append({'valid_i': int(i), 'neutral_word': nw, 'neutral_prompt': prompt, 'neutral_score': float(score)})
    neutral = pd.DataFrame(neutral_rows)

    item_effect_rows = []
    for i, row in valid.iterrows():
        sub = neutral[neutral['valid_i'] == i]
        item_effect_rows.append({
            'valid_i': int(i),
            'conflict_score': float(conflict_scores[i]),
            'neutral_mean': float(sub['neutral_score'].mean()),
            'effect': float(sub['neutral_score'].mean() - conflict_scores[i]),
        })
    item_effects = pd.DataFrame(item_effect_rows)

    # --- Pick representative neutral prompt per item ---
    patch_rows = []
    for i, row in valid.iterrows():
        sub = neutral[neutral['valid_i'] == i].copy()
        mean = item_effects.loc[item_effects['valid_i'] == i, 'neutral_mean'].iloc[0]
        sub['abs_dev'] = (sub['neutral_score'] - mean).abs()
        chosen = sub.sort_values('abs_dev').iloc[0]
        clean_prompt = chosen['neutral_prompt']
        corrupt_prompt = row['conflict_prompt']
        patch_rows.append({
            'valid_i': int(i),
            'item_id': int(row['item_id']),
            'word': row['word'], 'target': row['target'], 'distractor': row['distractor'],
            'clean_prompt': clean_prompt, 'corrupt_prompt': corrupt_prompt,
            'neutral_word': chosen['neutral_word'],
            'len_clean': int(model.to_tokens(clean_prompt, prepend_bos=USE_BOS).shape[-1]),
            'len_corrupt': int(model.to_tokens(corrupt_prompt, prepend_bos=USE_BOS).shape[-1]),
        })
    patch_prompts = pd.DataFrame(patch_rows)

    # --- Same patch-item selection rule as main run ---
    eligible = np.where(
        (patch_prompts['len_clean'].values == patch_prompts['len_corrupt'].values) &
        (item_effects['effect'].values > 0)
    )[0].tolist()
    if len(eligible) < 10:
        eligible = np.where(patch_prompts['len_clean'].values == patch_prompts['len_corrupt'].values)[0].tolist()
    patch_idx = eligible[:min(PATCH_N_ITEMS, len(eligible))]
    patch_items = valid.iloc[patch_idx].reset_index(drop=True)
    patch_prompts = patch_prompts.iloc[patch_idx].reset_index(drop=True)
    n_items = len(patch_items)
    print(f'  n_patching_items: {n_items}')

    # --- Build tokens + caches ---
    clean_tokens = to_tokens(patch_prompts['clean_prompt'])
    corrupt_tokens = to_tokens(patch_prompts['corrupt_prompt'])
    target_ids = torch.tensor(patch_items['target_id'].astype(int).tolist(), device=DEVICE)
    distractor_ids = torch.tensor(patch_items['distractor_id'].astype(int).tolist(), device=DEVICE)
    with torch.inference_mode():
        clean_logits, clean_cache = model.run_with_cache(clean_tokens)
        corrupt_logits, _corrupt_cache = model.run_with_cache(corrupt_tokens)
    clean_t, clean_d, clean_diff = logit_parts(clean_logits[:, -1, :], target_ids, distractor_ids)
    corrupt_t, corrupt_d, corrupt_diff = logit_parts(corrupt_logits[:, -1, :], target_ids, distractor_ids)

    # --- Build position groups (subject / target / query positions per item) ---
    pos_groups = []
    seq_len = int(clean_tokens.shape[1])
    for i, row in patch_items.iterrows():
        c_clean = clean_tokens[i].tolist()
        c_corrupt = corrupt_tokens[i].tolist()
        changed = [j for j, (a, b) in enumerate(zip(c_clean, c_corrupt)) if int(a) != int(b)]
        definition_subject = [changed[0]] if len(changed) > 0 else []
        query_word = [changed[1]] if len(changed) > 1 else []
        target_subseq = cont_ids(row['target'])
        starts = find_subsequence(c_corrupt, target_subseq)
        definition_target = [p for p in expand_positions(starts, len(target_subseq)) if p < seq_len - 1]
        pos_groups.append({
            'definition_subject': definition_subject,
            'definition_target': definition_target,
            'query_word': query_word,
            'source_all': sorted(set(definition_subject + definition_target + query_word)),
        })

    # --- Hook factories ---
    def apply_vector_patch(act, src, group):
        act = act.clone()
        for b in range(act.shape[0]):
            ps = pos_groups[b][group]
            if len(ps):
                act[b, ps, ...] = src[b, ps, ...]
        return act
    def vector_patch_hook(group):
        def fn(act, hook):
            return apply_vector_patch(act, clean_cache[hook.name], group)
        return fn
    def random_vector_patch_hook(group, perm):
        def fn(act, hook):
            src = clean_cache[hook.name]
            act = act.clone()
            for b in range(act.shape[0]):
                ps = pos_groups[b][group]
                if len(ps):
                    act[b, ps, ...] = src[int(perm[b].item()), ps, ...]
            return act
        return fn
    def run_scores(hooks):
        with torch.inference_mode():
            logits = model.run_with_hooks(corrupt_tokens, return_type='logits', fwd_hooks=hooks)[:, -1, :]
        return logit_parts(logits, target_ids, distractor_ids)

    # --- Determine triplet-selected hook+layer ---
    best_layer = best_comp = None
    if summary_df is not None:
        row_s = summary_df[summary_df['model_key'] == spec['model_key']]
        if len(row_s):
            best_layer = int(row_s['best_source_layer'].iloc[0])
            best_comp = str(row_s['best_source_component'].iloc[0])
            print(f'  using summary CSV: best_source = blocks.{best_layer}.{best_comp}')
    if best_layer is None:
        # Fallback: run layer scan internally for source_all only.
        print('  no summary entry; running layer scan for source_all...')
        best_rec = -float('inf')
        for comp in RESIDUAL_COMPONENTS:
            for layer in tqdm(range(n_layers), desc=f'{spec["model_key"]} scan {comp}'):
                name = f'blocks.{layer}.{comp}'
                if name not in model.hook_dict:
                    continue
                if layer < MIN_MECHANISM_LAYER:
                    continue
                _, _, diff = run_scores([(name, vector_patch_hook('source_all'))])
                rec = recovery(clean_diff, corrupt_diff, diff)
                if np.isfinite(rec) and rec > best_rec:
                    best_rec = rec; best_layer = layer; best_comp = comp
        print(f'  layer scan found: blocks.{best_layer}.{best_comp} (recovery {best_rec:+.4f})')
    source_name = f'blocks.{best_layer}.{best_comp}'

    # --- Matched triplet baseline at the selected hook ---
    matched_t, matched_d, matched_diff = run_scores([(source_name, vector_patch_hook('source_all'))])
    matched_rec = recovery(clean_diff, corrupt_diff, matched_diff)
    print(f'  matched triplet recovery (re-computed): {matched_rec:+.4f}')

    # --- 50 random-perturbation patches with logit logging ---
    rows = []
    base_seed = SEED + 1000
    for pi in tqdm(range(N_RANDOM_PERMUTATIONS), desc=f'{spec["model_key"]} random patches'):
        gen = torch.Generator(device=DEVICE); gen.manual_seed(base_seed + pi)
        perm = torch.randperm(n_items, device=DEVICE, generator=gen)
        rand_t, rand_d, rand_diff = run_scores([(source_name, random_vector_patch_hook('source_all', perm))])
        for b in range(n_items):
            donor_idx = int(perm[b].item())
            rows.append({
                'model_key': spec['model_key'],
                'model': spec.get('label', spec['model_key']),
                'item_id': int(patch_items.iloc[b]['item_id']),
                'word': patch_items.iloc[b]['word'],
                'target': patch_items.iloc[b]['target'],
                'distractor': patch_items.iloc[b]['distractor'],
                'neutral_word': patch_prompts.iloc[b]['neutral_word'],
                'hook_name': source_name,
                'triplet_selected_layer': best_layer,
                'control_type': 'item_mismatched_clean',
                'perm': int(pi),
                'donor_perm_index': donor_idx,
                'donor_item_id': int(patch_items.iloc[donor_idx]['item_id']),
                'donor_word': patch_items.iloc[donor_idx]['word'],
                'donor_target': patch_items.iloc[donor_idx]['target'],
                'm_clean': float(clean_diff[b]),
                'm_corrupt': float(corrupt_diff[b]),
                'm_matched': float(matched_diff[b]),
                'm_patched': float(rand_diff[b]),
                'target_logit_delta': float(rand_t[b] - corrupt_t[b]),
                'distractor_logit_delta': float(rand_d[b] - corrupt_d[b]),
            })

    # --- Free GPU memory before next model ---
    del model, clean_cache, _corrupt_cache, clean_logits, corrupt_logits, clean_tokens, corrupt_tokens
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)

print('Worker function ready.')

In [ ]:
# ============================================================
# 8. Run for all 5 models, save CSV incrementally
# ============================================================
ENABLED = [s for s in MECHANISM_MODEL_SPECS if s.get('enabled', True)]
print('Running:', [s['model_key'] for s in ENABLED])

all_dfs = []
out_csv = OUT_DIR / 'mechanism_random_controls_logit_decomp.csv'

for spec in ENABLED:
    try:
        df_part = run_random_logit_decomp_for_model(spec, summary_df)
        all_dfs.append(df_part)
        pd.concat(all_dfs, ignore_index=True).to_csv(out_csv, index=False)
        print(f'  wrote {out_csv}  (total rows: {sum(len(d) for d in all_dfs)})')
    except Exception as e:
        print(f'  FAILED on {spec["model_key"]}: {repr(e)}')

if all_dfs:
    result = pd.concat(all_dfs, ignore_index=True)
    print(f'\nDone. Total rows: {len(result)}')
    print(result.groupby('model_key').size())

In [ ]:
# ============================================================
# 9. Aggregate and print: MATCHED vs SWAP vs RANDOM logit decomp
# ============================================================
# This cell expects the matched and swap CSVs from the main reproduction run.
# Set the paths below; if either is missing, the matched/swap columns will be
# skipped and we'll print only the new random-control aggregation.
MATCHED_CSV = None  # e.g. Path('.../csv/mechanism_logit_decomposition.csv')
SWAP_CSV = None     # e.g. Path('.../csv/mechanism_definition_target_swap_controls.csv')

rand = pd.read_csv(out_csv)
rand_agg = rand.groupby('model_key').agg(
    rand_dtgt=('target_logit_delta', 'mean'),
    rand_ddis=('distractor_logit_delta', 'mean'),
    n_obs=('item_id', 'size'),
).reset_index()
rand_agg['rand_dmargin'] = rand_agg['rand_dtgt'] - rand_agg['rand_ddis']

merged = rand_agg.copy()
if MATCHED_CSV is not None and Path(MATCHED_CSV).exists():
    m = pd.read_csv(MATCHED_CSV)[['model_key', 'target_logit_delta', 'distractor_logit_delta', 'logit_diff_delta']]
    m.columns = ['model_key', 'matched_dtgt', 'matched_ddis', 'matched_dmargin']
    merged = merged.merge(m, on='model_key', how='left')
if SWAP_CSV is not None and Path(SWAP_CSV).exists():
    s = pd.read_csv(SWAP_CSV).groupby('model_key').agg(
        swap_dtgt=('target_logit_delta', 'mean'),
        swap_ddis=('distractor_logit_delta', 'mean'),
    ).reset_index()
    s['swap_dmargin'] = s['swap_dtgt'] - s['swap_ddis']
    merged = merged.merge(s, on='model_key', how='left')

order = ['qwen25_15b_base', 'qwen25_15b_instruct', 'gemma2_2b_base', 'gemma2_2b_instruct', 'olmo_1b_base']
merged['model_key'] = pd.Categorical(merged['model_key'], categories=order, ordered=True)
merged = merged.sort_values('model_key')

print('Logit decomposition comparison (means)')
print('=' * 130)
header = f'{"Model":<22}'
if 'matched_dtgt' in merged.columns:
    header += f' {"M.Δℓt":>9} {"M.Δℓd":>9} {"M.Δm":>9}  '
if 'swap_dtgt' in merged.columns:
    header += f' {"S.Δℓt":>9} {"S.Δℓd":>9} {"S.Δm":>9}  '
header += f' {"R.Δℓt":>9} {"R.Δℓd":>9} {"R.Δm":>9}'
print(header)
print('-' * 130)
for _, r in merged.iterrows():
    line = f'{str(r["model_key"]):<22}'
    if 'matched_dtgt' in merged.columns:
        line += f' {r["matched_dtgt"]:>+9.2f} {r["matched_ddis"]:>+9.2f} {r["matched_dmargin"]:>+9.2f}  '
    if 'swap_dtgt' in merged.columns:
        line += f' {r["swap_dtgt"]:>+9.2f} {r["swap_ddis"]:>+9.2f} {r["swap_dmargin"]:>+9.2f}  '
    line += f' {r["rand_dtgt"]:>+9.2f} {r["rand_ddis"]:>+9.2f} {r["rand_dmargin"]:>+9.2f}'
    print(line)

merged.to_csv(OUT_DIR / 'mechanism_logit_decomp_comparison.csv', index=False)
print(f'\nComparison table saved to: {OUT_DIR / "mechanism_logit_decomp_comparison.csv"}')

## Interpretation

Three conditions, three rows of `Δℓt / Δℓd / Δm` (matched vs swap vs random):

- **Matched (M):** source positions patched with the item's OWN clean activation. Binding preserved. Margin gains: +1.74 to +4.08 across the 5 models.
- **Swap (S):** identity match preserved, only `definition_target` swapped with a donor item. Distractor still drops (ΔΔℓd ≈ −2 vs matched), target collapses much more, margin reverses to negative.
- **Random (R, this notebook):** ALL three source positions taken from a donor item. No identity match, no local value.

**Key call for §5.3:**
- If `R.Δℓd` is strongly NEGATIVE (similar to matched/swap), distractor suppression is a generic patching side effect; matched triplet's unique action is target protection.
- If `R.Δℓd` is near zero or POSITIVE, distractor suppression correlates with item-identity match; current §5.3 framing stands.